In [0]:
from pyspark.sql.functions import sha2, concat_ws, col

def check_full_integrity():
    core_cols = ["user_id", "event_type", "product_id", "category_id", "price"]
    
    # 1. Generate hashes for the entire Bronze and Silver populations
    bronze_hashes = spark.table("workspace.dev_bronze_layer.events_raw") \
        .select(sha2(concat_ws("||", *core_cols), 256).alias("row_hash"))
        
    silver_hashes = spark.table("workspace.dev_silver_layer.events_cleaned") \
        .select(sha2(concat_ws("||", *core_cols), 256).alias("row_hash"))

    # 2. Use a Left Anti Join to find ANY Silver hash not in Bronze
    # An anti join returns rows from the left (Silver) that have no match in the right (Bronze)
    mismatched_rows = silver_hashes.join(bronze_hashes, "row_hash", "left_anti")
    
    mismatch_count = mismatched_rows.count()

    # 3. Final Validation
    if mismatch_count == 0:
        print(f"Success: All {silver_hashes.count():,} rows verified. 100% integrity.")
    else:
        print(f"ERROR: {mismatch_count} rows in Silver could not be traced to Bronze!")
        # Optional: show a few samples of the orphans for debugging
        mismatched_rows.show(5, truncate=False)
        raise Exception("Row-level integrity failed for the full dataset.")

check_full_integrity()